# 🚀 AI Multi-Docs Extraction Pipeline: Walkthrough & Real Data Runner

สมุดบันทึก (Jupyter Notebook) สำหรับรันกระบวนการสกัดและประมวลผลเอกสารทั้งระบบแบบ **End-to-End ด้วยข้อมูลจริงจาก `pipeline_storage`**
ช่วยให้สามารถรันและตรวจสอบการไหลของข้อมูลจริงในแต่ละขั้นตอน (Step-by-Step Execution & Observability) ได้อย่างสมบูรณ์

## 🛠️ Step 0: ตั้งค่า Working Directory, Environment และนำเข้า Pipeline Services

In [ ]:
import os
import sys
import glob
import json
import pandas as pd
from IPython.display import display, JSON
from dotenv import load_dotenv

# 1. Ensure working directory is set to project root
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f'📂 Project Root Working Directory: {os.getcwd()}')

# 2. Load Environment Variables & Pipeline Services
load_dotenv()
from src.core.pipeline import (
    run_init,
    run_split_and_match,
    run_extract,
    run_transform_to_db,
    run_validate,
    run_export_outputs,
    run_pipeline_all,
    reset_pipeline_data
)
from src.core.config_loader import load_system_settings, get_default_doc_type
from src.core.storage_manager import storage_manager

DOC_TYPE = get_default_doc_type()
print(f"✅ Pipeline Services Ready. Active Target Doc Type: '{DOC_TYPE}'")


## 🧹 Step 0.1: (Optional) รีเซ็ตฐานข้อมูลและล้างไฟล์ชั่วคราว (Fresh Start)
กดรันเซลล์นี้เมื่อต้องการ **ล้างประวัติเอกสารใน SQLite** และ **ล้างไฟล์ชั่วคราวใน `03_preprocess/` และ `04_processing/`** เพื่อเริ่มทดสอบใหม่ตั้งแต่ต้น

In [ ]:
# ปลดล็อคหรือกดรันเซลล์นี้เพื่อเคลียร์ข้อมูลเดิมก่อนเริ่มรันใหม่
reset_result = reset_pipeline_data(doc_type=DOC_TYPE, clear_storage_temp=True, clear_database=True)
print("🧹 Reset Pipeline Data Result:", reset_result)
print("🎉 Ready for a brand new clean run!")

## ⚙️ Step 1: System Initialization (`Run_01`)
ตรวจสอบความพร้อมของ `settings.json`, Schema ฐานข้อมูล SQLite, และโครงสร้างโฟลเดอร์ใน `storage/`

In [ ]:
print("--- [Stage 1] Initializing System & Validating Environment ---")
# หมายเหตุ: หากต้องการเริ่มต้นระบบใหม่แบบ Clean 100% (Drop & Recreate Tables ทุกตาราง): ให้เปิดใช้งานบรรทัดด้านล่างนี้
# init_success = run_init(drop_and_recreate=True)
init_success = run_init(drop_and_recreate=False)
if init_success:
    print("🎉 System is READY and all storage folders & DB tables are verified!")
else:
    print("❌ System initialization encountered errors. Please check configs or .env")

## 📄 Step 2: Split PDFs & Match Merchant Sources (`Run_02`)
อ่านไฟล์เอกสารจริงจาก `01_drop_zone/` (รองรับทั้ง PDF และภาพ `.jpg`, `.png`, `.webp`) เพื่อ:
1. ตรวจจับร้านค้า (Merchant Matching: SPX, Grab, Shopee, 7-Eleven)
2. ตัดหน้า PDF เป็นไฟล์ภาพ `.jpg` ลงใน `03_preprocess/`
3. ลงทะเบียน Batch & Pages เข้าสู่ฐานข้อมูล

In [ ]:
print("--- [Stage 2] Processing Inbox Documents and Matching Merchant Sources ---")
split_results = run_split_and_match(doc_type=DOC_TYPE)

if split_results:
    print(f"\n✅ Successfully processed {len(split_results)} document batch(es):")
    for res in split_results:
        print(f"\n📦 Batch ID: {res.get('batch_id')}")
        print(f"   - Original File: {res.get('filename')}")
        print(f"   - Matched Merchant: {res.get('matched_source')}")
        print(f"   - Total Pages: {res.get('total_pages')}")
        print(f"   - Split Page Images ({len(res.get('page_images', []))} files):")
        for img in res.get('page_images', []):
            print(f"     🖼️ {img}")
else:
    print("ℹ️ No new document files found in drop zone. (You can place PDF/JPG files into storage/companies/C00000_SAMPLE/expense_receipt/01_drop_zone/Upload/)")

## 🤖 Step 3: AI Document Extraction (`Run_03`)
นำรูปภาพหน้าที่ตัดแล้วใน `03_preprocess/` ส่งให้ AI สกัดข้อมูลตามโครงสร้าง `extract-schema.json`
และบันทึกไฟล์ JSON ที่สกัดได้ลงใน `04_processing/extracted/`

In [ ]:
print('--- [Stage 3] Extracting Document Data with AI ---')
extract_result = run_extract(doc_type=DOC_TYPE)
print(f'📊 Extraction Summary: {extract_result}')

# Preview extracted JSON payload files
queue_pattern = f"{storage_manager.get_extracted_dir('C00000_SAMPLE', DOC_TYPE)}/**/*.json"
queue_files = glob.glob(queue_pattern, recursive=True)
if queue_files:
    print(f'\n💾 Found {len(queue_files)} extracted JSON file(s):')
    sample_file = queue_files[0]
    print(f'   Showing preview from: {sample_file}')
    with open(sample_file, 'r', encoding='utf-8') as jf:
        sample_json = json.load(jf)
    display(JSON(sample_json))
else:
    print('ℹ️ No JSON files found in extracted storage.')


## 🛡️ Step 4: Validate & Post-Process Data (`Run_04`)
ตรวจสอบความถูกต้องของข้อมูลจริงใน `04_processing/extracted/`:
- ตรวจสอบเลขประจำตัวผู้เสียภาษี (Tax ID)
- ปรับรูปแบบวันที่ปี พ.ศ. ➔ ค.ศ. (BE to AD Normalization)
- ตรวจสอบสูตรการเงิน (Subtotal - Discount + VAT == Net)
- ตรวจสอบผลรวมรายการสินค้าเทียบกับ Subtotal
- จัดลำดับความสำคัญ (Review Priority: HIGH/MED/LOW) และกำหนดสถานะ `PROCESSED` หรือ `NEEDS_REVIEW`

In [ ]:
print("--- [Stage 4] Validating and Post-Processing Queue Records ---")
validate_result = run_validate(doc_type=DOC_TYPE)
print(f"📊 Validation Summary: {validate_result}")

## 💾 Step 5: Transform Data to Relational SQLite Database (`Run_05`)
นำเข้าข้อมูลที่ผ่านการตรวจสอบแล้วเข้าสู่ฐานข้อมูล SQLite ในตาราง `documents`, `expense_receipts`, และ `receipt_items`

In [ ]:
print("--- [Stage 5] Importing Records into Relational Database ---")
db_result = run_transform_to_db(doc_type=DOC_TYPE)
print(f"📊 DB Transformation Summary: {db_result}")

## 📊 Step 6: Generate Output Reports (`Run_06`)
สร้างรายงานสรุปผลลัพธ์ผ่าน Exporters ทั้งหมด (Google Sheet Summary, Accounting Line Items, Express PV Voucher)

In [ ]:
print('--- [Stage 6/Export] Generating Output Reports (CSV / JSON / Express PV) ---')
export_result = run_export_outputs(doc_type=DOC_TYPE)
print(f'📊 Export Summary: {export_result}')

# Display generated outputs
out_dir = storage_manager.get_output_dir('C00000_SAMPLE', DOC_TYPE)
csv_exports = glob.glob(f'{out_dir}/*.csv')
for csv_file in csv_exports:
    print(f'\n--------------------------------------------------------')
    print(f'📄 Output Report: {csv_file}')
    print(f'--------------------------------------------------------')
    try:
        encoding = 'cp874' if 'express_pv' in csv_file else 'utf-8-sig'
        df_report = pd.read_csv(csv_file, encoding=encoding)
        display(df_report.head(10))
    except Exception as read_err:
        print(f'Could not preview CSV: {read_err}')


## ⚡ Step 7: (Optional) Full Pipeline Single-Command Execution (`Run All`)
รันทุกขั้นตอนตั้งแต่ต้นจนจบ (Stage 1 ➔ Stage 6) ในคำสั่งเดียว

In [ ]:
# ปลด comment เมื่อต้องการสั่งรันทุก Stage พร้อมกันแบบ One-Shot
# print("🚀 Executing Full Pipeline End-to-End...")
# full_pipeline_result = run_pipeline_all(doc_type=DOC_TYPE)
# print("🎉 Pipeline Run Completed:", full_pipeline_result)